# Intro to LLMs — Prompts, Roles, and Conversations

This notebook isn't about stocks. It's about the actual mechanics underneath
every LLM feature you'll build from here on — the news sentiment panel, the
dashboard chat agent, everything. Get comfortable here first.

**One package for all three providers.** OpenAI, Anthropic, and Gemini each
publish an "OpenAI-compatible" endpoint, so the `openai` Python package can
talk to any of them — you only ever change the API key, the `base_url`, and
the model name. No `anthropic` package, no `google.generativeai` package
(which is deprecated anyway — if you've seen that warning, this sidesteps it
completely).

**Three things you'll actually understand by the end:**
1. What a "role" is, and why `system` / `user` / `assistant` exist as separate things.
2. That an LLM has **no memory** — "conversation" is an illusion you build yourself.
3. That a system prompt is **optional** — a lever you choose to pull, not something every call needs.


## Setup
Free key: https://aistudio.google.com/apikey — then set `GEMINI_API_KEY`.

## 1. Setup — `.env` and picking a provider

```
OPENAI_API_KEY=sk-...
ANTHROPIC_API_KEY=sk-ant-...
GEMINI_API_KEY=...
```

**Honest note on cost:** OpenAI and Anthropic both require a payment method on
file for real usage — neither has a lasting free tier the way Gemini does
(1,500 free requests/day, no card required). If you don't have credits, use
Gemini, or set `PROVIDER = "mock"` and this notebook still runs end to end —
every real call gets replaced with a clearly-labeled fake response.

In [23]:
import os
from dotenv import load_dotenv
load_dotenv()

# ---- choose your provider here ----
PROVIDER = "gemini"   # one of: "openai", "anthropic", "gemini", "mock"

CONFIG = {
    "openai":    {"api_key": os.environ.get("OPENAI_API_KEY"),    "base_url": None,
                  "model": "gpt-5-mini"},
    "anthropic": {"api_key": os.environ.get("ANTHROPIC_API_KEY"), "base_url": "https://api.anthropic.com/v1/",
                  "model": "claude-sonnet-5"},
    "gemini":    {"api_key": os.environ.get("GEMINI_API_KEY"),    "base_url": "https://generativelanguage.googleapis.com/v1beta/openai/",
                  "model": "gemini-flash-latest"},   # alias -- always points at Google's current recommended flash model
}

print(f'provider: {PROVIDER}')

provider: gemini


## 2. The three roles

Every LLM chat API organizes a conversation as a list of messages, and every
message has a **role**:

- **`system`** — instructions about *how the model should behave*, set once.
  **This one is optional** — leave it out, and the model just uses its own
  default behavior. You'll see both cases below, back to back.
- **`user`** — what the person actually typed.
- **`assistant`** — what the model said back. You also use this role
  yourself when showing the model its own prior reply (section 5).

In [30]:
def chat(messages, system=None, provider=PROVIDER, max_tokens=1000, verbose=False):
    """
    Send a list of {"role":..., "content":...} messages, get back the reply text.
    `system` is genuinely optional -- pass nothing, and no system message is sent at all.

    verbose=True prints WHY the response ended -- "stop" (finished naturally)
    vs "length" (cut off by max_tokens). Use this instead of guessing whether
    a short or odd-looking answer is a truncation problem or something else.
    """
    if provider == "mock":
        last_user = next((m['content'] for m in reversed(messages) if m['role']=='user'), '')
        sys_note = f' [as instructed: {system[:40]}...]' if system else ' [no system prompt given]'
        return f"[MOCK REPLY]{sys_note} You said: \"{last_user[:60]}\" -- imagine a real, helpful answer here."

    from openai import OpenAI
    cfg = CONFIG[provider]
    client = OpenAI(api_key=cfg["api_key"], base_url=cfg["base_url"]) if cfg["base_url"] else OpenAI(api_key=cfg["api_key"])

    full_messages = ([{"role": "system", "content": system}] if system else []) + messages
    resp = client.chat.completions.create(model=cfg["model"], messages=full_messages, max_tokens=max_tokens)

    choice = resp.choices[0]
    if verbose:
        print(f"[finish_reason: {choice.finish_reason}]")
        if choice.finish_reason == "length":
            print(f"[TRUNCATED -- hit the {max_tokens}-token limit before finishing. Raise max_tokens.]")
    return choice.message.content

print('chat() ready')

chat() ready


## 3. A basic call — no system prompt, on purpose

The simplest possible use: a single user message, **nothing else**. No
`system=` argument is passed at all — watch the mock note confirm this
explicitly. This is what "just asking a question" looks like with zero
configuration.

In [31]:
reply = chat([{"role": "user", "content": "What's a good rule of thumb for saving money each month?"}], verbose=True)
print(reply)

[finish_reason: length]
[TRUNCATED -- hit the 1000-token limit before finishing. Raise max_tokens.]
The most widely recommended and effective rule of thumb is the **50/30/20 Rule**. 

It uses your **take-home pay** (after-tax income) as the baseline and divides it into three categories:

### 1. The 50/30/20 Rule
* **50% for Needs:** Rent/mortgage, utilities, groceries, transportation, insurance, and minimum debt payments.
* **30% for Wants:** Dining out, shopping, hobbies, streaming subscriptions, and travel.
* **20% for Savings & Debt Payoff:** High-yield savings, investments, retirement accounts, and paying off debt beyond the minimum payments.

---

### If 20% Feels Too High Right Now: "Pay Yourself First"
If saving 20% isn't realistic


**Notice there was no persona, no instruction, nothing shaping the answer** —
just whatever the model's own default "helpful assistant" behavior happens to
be. That's a completely valid way to call an LLM. The system prompt is
optional, not required — but it's also the single biggest lever you have
when you *do* want to shape the answer, which is exactly what the next
section shows.

## 4. Now, the same question — WITH a system prompt

Same user question as section 3, unchanged. The only thing that's different
this time: a `system=` argument is passed. Watch how much the *personality
and content* of the answer shifts, purely from that one addition.

In [ ]:
question = [{"role": "user", "content": "Should I invest my savings in the stock market?"}]

personas = {
    "Cautious financial advisor": "You are a cautious, risk-averse financial advisor. Always mention downside risk first.",
    "Enthusiastic startup founder": "You are an enthusiastic startup founder who thinks everyone should take more risks. Be energetic and brief.",
    "Terse, no-nonsense analyst": "You are a terse financial analyst. Answer in one blunt sentence, no pleasantries.",
}

for name, system_prompt in personas.items():
    print(f'--- {name} ---')
    print(chat(question, system=system_prompt))
    print()

**Same question, three completely different answers** — and compare all
three to section 3's answer, which had no system prompt at all. Four
variations of the same question, four different outcomes, and the *only*
thing that changed each time was whether (and how) a system prompt was set.

## 5. "Memory" is an illusion you build yourself

The model does not remember anything between calls. Every request is
completely stateless. "Conversation" only works because *you* resend the
entire history every time, with the newest message tacked on the end.

In [ ]:
conversation = []

def say(user_text, system=None):
    conversation.append({"role": "user", "content": user_text})
    reply = chat(conversation, system=system)
    conversation.append({"role": "assistant", "content": reply})
    print(f'YOU:  {user_text}')
    print(f'BOT:  {reply}')
    print()
    return reply

SYSTEM = "You are a friendly assistant helping someone plan a trip. Keep replies short."

say("I want to go somewhere warm in December.", system=SYSTEM)
say("Somewhere in Africa specifically.", system=SYSTEM)
say("What did I say I wanted, again?", system=SYSTEM)

In [ ]:
import json
print(json.dumps(conversation, indent=2))

**That printed list is the whole trick.** No hidden state on the server —
just a growing list of messages resent in full, every time.

**Try it yourself:** comment out the line that appends the assistant's reply,
ask a follow-up that depends on earlier context, and watch the model lose the
thread completely — because as far as it knows, this is the first thing
anyone has ever said to it.